In [1]:
import sys
sys.path.append("../")
from plotting import CandlePlot
import pandas as pd

In [2]:
from technicals.patterns import apply_patterns

In [3]:
df = pd.read_pickle("../data/GBP_JPY_H1.pkl")

In [4]:
df_an = df[['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c']].copy()

In [5]:
df_an.tail()

,time,mid_o,mid_h,mid_l,mid_c
54517,2024-10-09 19:00:00+00:00,195.142,195.194,195.062,195.083
54518,2024-10-09 20:00:00+00:00,195.082,195.242,195.016,195.200
54519,2024-10-09 21:00:00+00:00,195.072,195.085,194.878,194.990
54520,2024-10-09 22:00:00+00:00,194.986,195.092,194.948,194.960
54521,2024-10-09 23:00:00+00:00,194.959,195.060,194.873,194.906


In [6]:
#direction = df_an.mid_c - df_an.mid_o
#body_size = abs(direction)
#direction = [1 if x>=0 else 0 for x in direction ]
#full_range = df_an.mid_h - df_an.mid_l
#body_perc = (body_size / full_range) * 100
#body_lower = df_an[['mid_c', 'mid_o']].min(axis=1)
#body_upper = df_an[['mid_c', 'mid_o']].max(axis=1)
#body_bottom_perc = ((body_lower - df_an.mid_l) / full_range) * 100
#body_top_perc = ((body_upper - df_an.mid_l) / full_range) * 100

In [7]:
#df_an['body_upper'] = body_upper
#df_an['body_lower'] = body_lower
#df_an['body_perc'] = body_perc
#df_an['body_bottom_perc'] = body_bottom_perc
#df_an['body_top_perc'] = body_top_perc
#df_an['full_range'] = full_range

In [8]:
df_an = apply_patterns(df_an)

In [9]:
HANGING_MAN_BODY = 15.0
HANGING_MAN_HEIGHT = 75.0
SHOOTING_STAR_HEIGHT = 25.0
SPINNING_TOP_MIN = 40.0
SPINNING_TOP_MAX = 60.0
MARUBOZU = 98.0
ENGULFING_FACTOR = 1.1

apply_marubozu = lambda x: x.body_perc > MARUBOZU

def apply_hanging_man(row):
    if row.body_bottom_perc > HANGING_MAN_HEIGHT:
        if row.body_perc < HANGING_MAN_BODY:
            return True
    return False

def apply_shooting_star(row):
    if row.body_top_perc < SHOOTING_STAR_HEIGHT:
        if row.body_perc < HANGING_MAN_BODY:
            return True
    return False

def apply_spinning_top(row):
    if row.body_top_perc < SPINNING_TOP_MAX:
        if row.body_bottom_perc > SPINNING_TOP_MIN:
            if row.body_perc < HANGING_MAN_BODY:
                return True
    return False

def apply_engulfing(row):
    if row.direction != row.direction_prev:
        if row.body_size > row.body_size_prev * ENGULFING_FACTOR:
            return True
    return False

TWEEZER_BODY = 15.0
TWEEZER_HL = 0.01
TWEEZER_TOP_BODY = 40.0
TWEEZER_BOTTOM_BODY = 60.0

def apply_tweezer_top(row):
    if abs(row.body_size_change) < TWEEZER_BODY:
        if row.direction == -1 and row.direction != row.direction_prev:
            if abs(row.low_change) < TWEEZER_HL and abs(row.high_change) < TWEEZER_HL:
                if row.body_top_perc < TWEEZER_TOP_BODY:
                    return True
    return False               

def apply_tweezer_bottom(row):
    if abs(row.body_size_change) < TWEEZER_BODY:
        if row.direction == 1 and row.direction != row.direction_prev:
            if abs(row.low_change) < TWEEZER_HL and abs(row.high_change) < TWEEZER_HL:
                if row.body_bottom_perc > TWEEZER_BOTTOM_BODY:
                    return True
    return False     

MORNING_STAR_PREV2_BODY = 90.0
MORNING_STAR_PREV_BODY = 10.0

def apply_morning_star(row, direction=1):
    if row.body_perc_prev_2 > MORNING_STAR_PREV2_BODY:
        if row.body_perc_prev < MORNING_STAR_PREV_BODY:
            if row.direction == direction and row.direction_prev_2 != direction:
                if direction == 1:
                    if row.mid_c > row.mid_point_prev_2:
                        return True
                else:
                    if row.mid_c < row.mid_point_prev_2:
                        return True
    return False

In [10]:
df_an['body_size_prev'] = df_an.body_size.shift(1)
df_an['direction_prev'] = df_an.direction.shift(1)
df_an['direction_prev_2'] = df_an.direction.shift(2)
df_an['body_perc_prev'] = df_an.body_perc.shift(1)
df_an['body_perc_prev_2'] = df_an.body_perc.shift(2)
df_an['HANGING_MAN'] = df_an.apply(apply_hanging_man, axis=1)
df_an['SHOOTING_STAR'] = df_an.apply(apply_shooting_star, axis=1)
df_an['SPINNING_TOP'] = df_an.apply(apply_spinning_top, axis=1)
df_an['MARUBOZU'] = df_an.apply(apply_marubozu, axis=1)
df_an['ENGULFING'] = df_an.apply(apply_engulfing, axis=1)
df_an['TWEEZER_TOP'] = df_an.apply(apply_tweezer_top, axis=1)
df_an['TWEEZER_BOTTOM'] = df_an.apply(apply_tweezer_bottom, axis=1)
df_an['MORNING_STAR'] = df_an.apply(apply_morning_star, axis=1)
df_an['EVENING_STAR'] = df_an.apply(apply_morning_star, axis=1, direction=-1)

In [18]:
df_an[df_an['MORNING_STAR'] == True]

,time,mid_o,mid_h,mid_l,mid_c,body_lower,body_upper,body_bottom_perc,body_top_perc,body_perc,...,body_perc_prev_2,HANGING_MAN,SHOOTING_STAR,SPINNING_TOP,MARUBOZU,ENGULFING,TWEEZER_TOP,TWEEZER_BOTTOM,MORNING_STAR,EVENING_STAR
1058,2016-03-09 02:00:00+00:00,159.660,159.884,159.560,159.870,159.660,159.870,30.864198,95.679012,64.814815,...,90.086207,False,False,False,False,True,False,False,True,False
9085,2017-06-22 14:00:00+00:00,140.676,141.128,140.640,140.984,140.676,140.984,7.377049,70.491803,63.114754,...,91.777778,False,False,False,False,False,False,False,True,False
16610,2018-09-07 01:00:00+00:00,142.801,143.024,142.779,142.991,142.801,142.991,8.979592,86.530612,77.551020,...,90.235690,False,False,False,False,True,False,False,True,False
23230,2019-10-01 16:00:00+00:00,131.852,132.896,131.792,132.662,131.852,132.662,5.434783,78.804348,73.369565,...,94.313725,False,False,False,False,False,False,False,True,False
25802,2020-03-02 23:00:00+00:00,138.109,138.572,138.109,138.551,138.109,138.551,0.000000,95.464363,95.464363,...,91.612903,False,False,False,False,True,False,False,True,False
28631,2020-08-13 20:00:00+00:00,139.550,139.754,139.515,139.736,139.550,139.736,14.644351,92.468619,77.824268,...,95.846645,False,False,False,False,True,False,False,True,False
28687,2020-08-18 04:00:00+00:00,138.698,138.802,138.696,138.768,138.698,138.768,1.886792,67.924528,66.037736,...,92.307692,False,False,False,False,True,False,False,True,False
31465,2021-01-28 23:00:00+00:00,143.092,143.173,143.077,143.156,143.092,143.156,15.625000,82.291667,66.666667,...,95.774648,False,False,False,False,False,False,False,True,False
33455,2021-05-25 20:00:00+00:00,153.810,153.954,153.796,153.936,153.810,153.936,8.860759,88.607595,79.746835,...,91.803279,False,False,False,False,True,False,False,True,False
35208,2021-09-05 21:00:00+00:00,152.118,152.277,152.075,152.119,152.118,152.119,21.287129,21.782178,0.495050,...,92.929293,False,True,False,False,False,False,False,True,False


In [12]:
import plotly.graph_objects as go

In [20]:
MARKER = '#0066FF'
dfp = df_an.iloc[1000:1200]
cp = CandlePlot(dfp, candles=True)
df_temp = cp.df_plot[cp.df_plot.MORNING_STAR==True]
cp.fig.add_trace(go.Candlestick(
                x=df_temp.sTime,
                open=df_temp.mid_o,
                high=df_temp.mid_h,
                low=df_temp.mid_l,
                close=df_temp.mid_c,
                line=dict(width=1), opacity=1,
                increasing_fillcolor=MARKER,
                decreasing_fillcolor=MARKER,
                increasing_line_color=MARKER,  
                decreasing_line_color=MARKER
            ))

cp.show_plot()

: 